In [2]:
!rm -rf $HOME/.local/share/Trash/files

In [3]:
!conda install -y -c conda-forge gdal

Retrieving notices: done
Channels:
 - conda-forge
 - nvidia
 - pytorch
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.3.0
    latest version: 25.5.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /home/ec2-user/anaconda3/envs/python3

  added / updated specs:
    - gdal


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2025.7.14  |       hbd8a1cb_0         152 KB  conda-forge
    certifi-2025.7.14          |     pyhd8ed1ab_0         156 KB  conda-forge
    freexl-2.0.0               |       h9dce30a_2          58 KB  conda-forge
    gdal-3.11.0                |  py310he9751b3_5         1.6 MB  conda-forge
    geos-3.13.1                |       h97f6797_0         1.8 MB  conda-forge
    json-c-0.18                |      

In [4]:
import os
import json
from osgeo import ogr

In [5]:
import os
import json
from osgeo import ogr

os.environ["OGR_GEOJSON_MAX_OBJ_SIZE"] = "0"  # No limit

# Input and output files
input_file = "nts_250k_july_25.geojson"
output_file = "nts_opensearch_bulk.json"

# Open input GeoJSON
driver = ogr.GetDriverByName("GeoJSON")
data_source = driver.Open(input_file, 0)  # 0 = read-only
layer = data_source.GetLayer()

bulk_docs = []

def round_coords(geom_json, precision=6):
    def round_list(coords):
        if isinstance(coords[0], list):
            return [round_list(c) for c in coords]
        else:
            return [round(c, precision) for c in coords]
    geom_json["coordinates"] = round_list(geom_json["coordinates"])
    return geom_json

def should_simplify(geom_json, size_limit_bytes=9800000):
    size = len(json.dumps(geom_json).encode("utf-8"))
    return size > size_limit_bytes

for feature in layer:
    geom = feature.GetGeometryRef()
    properties = feature.items()

    if geom is None:
        continue

    # Extract bounding box as envelope type for OpenSearch
    envelope = geom.GetEnvelope()
    bbox_envelope = {
        "type": "envelope",
        "coordinates": [
            [envelope[0], envelope[3]],  # top-left: minX, maxY
            [envelope[1], envelope[2]]   # bottom-right: maxX, minY
        ]
    }

    if geom.GetGeometryType() == ogr.wkbMultiPolygon:
        for i in range(geom.GetGeometryCount()):
            polygon = geom.GetGeometryRef(i)
            geom_json = json.loads(polygon.ExportToJson())

            if should_simplify(geom_json):
                print(f"Simplifying oversized geometry for CFSAUID {properties.get('CFSAUID')}")
                simplified_geom = polygon.Simplify(0.0002)
                geom_json = json.loads(simplified_geom.ExportToJson())

            doc = {
                "name": properties.get("CFSAUID", "Unknown"),
                "attributes": properties,
                "bbox": bbox_envelope,
                "location": geom_json
            }
            bulk_docs.append(doc)

    elif geom.GetGeometryType() == ogr.wkbPolygon:
        geom_json = json.loads(geom.ExportToJson())

        if should_simplify(geom_json):
            print(f"Simplifying oversized geometry for CFSAUID {properties.get('CFSAUID')}")
            simplified_geom = geom.Simplify(0.0002)
            geom_json = json.loads(simplified_geom.ExportToJson())

        doc = {
            "name": properties.get("CFSAUID", "Unknown"),
            "attributes": properties,
            "bbox": bbox_envelope,
            "location": geom_json
        }
        bulk_docs.append(doc)

# Write to OpenSearch bulk format
with open(output_file, "w") as f:
    for doc in bulk_docs:
        f.write(json.dumps({ "index": { "_index": "nts-index" } }) + "\n")
        f.write(json.dumps(doc) + "\n")

print(f"Exported {len(bulk_docs)} documents to {output_file} with envelope bbox format.")

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/osgeo/gdal.py:330: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Exported 1243 documents to nts_opensearch_bulk.json with envelope bbox format.


Code below will authenticate with OpenSearch endpoint using the SageMaker IAM role

In [6]:
!pip install -q boto3
!pip install -q requests
!pip install -q requests-aws4auth
!pip install -q opensearch-py

In [7]:
import boto3
import requests
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
from requests_aws4auth import AWS4Auth
import opensearch

In [8]:
from opensearch import *
region = "ca-central-1"
aos_host = "search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com"
os_secret_id = "OpenSearchSecret-geocore-semantic-search-with-opensearch-stage"

#awsauth = get_awsauth_from_secret(region, secret_id=os_secret_id)
#awsauth = ("admin", "Semantic123!")
credentials = boto3.Session().get_credentials()
awsauth = AWS4Auth(credentials.access_key, credentials.secret_key, region, 'es', session_token=credentials.token)
    
aos_client =create_opensearch_connection(aos_host, awsauth)


Connection to OpenSearch established: <OpenSearch([{'host': 'search-semantic-search-arieibeskhrn6vn2qd7gf5br7q.ca-central-1.es.amazonaws.com', 'port': 443}])>


In [10]:
index_name = "nts-index"
index = {
  "mappings": {
    "properties": {
      "CFSAUID":   { "type": "keyword" }, 
      "DGUID":     { "type": "keyword" },
      "PRUID":     { "type": "keyword" },
      "PRNAME":    { "type": "text" },    
      "LANDAREA":  { "type": "float" },
      "LONG":      { "type": "float" },
      "LAT":       { "type": "float" },
      "bbox":      { "type": "geo_shape" },
      "location":  { "type": "geo_shape" } 
    }
  }
}

In [11]:
delete_aos_index_if_exists(aos_client, index_to_delete=index_name)

Current indexes: ['id-audit-logs', 'minilm-pretrain-knn', 'semantic-search-audit-logs', 'minilm-knn-2', '.ql-datasources', '.kibana_92668751_admin_1', 'minilm-knn-multilingual', 'vcs-audit-logs', '.opendistro-reports-definitions', '.kibana_1', '.opendistro_security', '.opendistro-reports-instances', 'minilm-knn', 'vcs-audit-logs-v2', '.plugins-ml-config', '.opensearch-observability', 'audit-logs-stage', 'opensearch_dashboards_sample_data_logs', 'geolocator-audit-logs', 'opensearch_dashboards_sample_data_flights', '.opendistro-job-scheduler-lock', 'fsa-postal-code-index', 'mpnet-mpf-knn']
Index nts-index does not exist.
Indexes after deletion attempt: ['id-audit-logs', 'minilm-pretrain-knn', 'semantic-search-audit-logs', 'minilm-knn-2', '.ql-datasources', '.kibana_92668751_admin_1', 'minilm-knn-multilingual', 'vcs-audit-logs', '.opendistro-reports-definitions', '.kibana_1', '.opendistro_security', '.opendistro-reports-instances', 'minilm-knn', 'vcs-audit-logs-v2', '.plugins-ml-config', 

In [12]:
aos_client.indices.create(index=index_name,body=index,ignore=400)

{'acknowledged': True, 'shards_acknowledged': True, 'index': 'nts-index'}

In [13]:
import os
import requests
import time
from requests.exceptions import RequestException

MAX_BYTES = 10_485_760  # 10 MB limit
bulk_file = "nts_opensearch_bulk.json"

with open(bulk_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}")

buffer = []
buffer_bytes = 0
doc_count = 0
chunk_number = 1

def send_chunk(buffer, chunk_number, start_doc, end_doc):
    data = ''.join(buffer)
    try:
        resp = requests.post(
            f"https://{aos_host}/_bulk",
            auth=awsauth,
            headers={"Content-Type": "application/x-ndjson"},
            data=data
        )
        if resp.status_code != 200:
            print(f"Chunk {chunk_number} ({start_doc}-{end_doc}): Status {resp.status_code}")
            print("Response:", resp.text)
        else:
            print(f"Chunk {chunk_number} ({start_doc}-{end_doc}): Uploaded")
            response_json = resp.json()
            if response_json.get("errors"):
                print("Some items failed in this chunk.")
                for idx, item in enumerate(response_json["items"]):
                    action = next(iter(item))
                    result = item[action]
                    if result.get("error"):
                        print(f"Doc {start_doc + idx}: {result['error']['type']} - {result['error']['reason']}")
    except RequestException as e:
        print(f"Chunk {chunk_number} ({start_doc}-{end_doc}): Request error: {e}")
    time.sleep(0.2)

i = 0
while i < len(lines):
    index_line = lines[i]
    doc_line = lines[i + 1]
    pair = index_line + doc_line
    pair_size = len(pair.encode("utf-8"))

    if buffer_bytes + pair_size > MAX_BYTES:
        # Send the current buffer
        start_doc = doc_count - len(buffer) // 2 + 1
        end_doc = doc_count
        send_chunk(buffer, chunk_number, start_doc, end_doc)
        buffer = []
        buffer_bytes = 0
        chunk_number += 1

    buffer.append(index_line)
    buffer.append(doc_line)
    buffer_bytes += pair_size
    doc_count += 1
    i += 2

# Send any remaining buffer
if buffer:
    start_doc = doc_count - len(buffer) // 2 + 1
    end_doc = doc_count
    send_chunk(buffer, chunk_number, start_doc, end_doc)

Total lines: 2486
Chunk 1 (1-1243): Uploaded
